# No double-booking — step by step

A hands-on lab. Run one cell at a time, read what happened, then move on. You will create a GiST exclusion constraint to prevent overlapping operations on the same machine, then prove it works by attempting a conflicting insert.

Each step shows the **plain SQL** too, so you can copy it and run it in psql, the Lakebase SQL editor, or any tool. The Python just runs that SQL against your branch.

**Before you start:** finish `00-setup/` so `LB_PROFILE`, `LB_PROJECT`, `LB_BRANCH`, `LB_SCHEMA` are set and the factory is loaded on your branch.

## Setup — connect to your branch

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..', 'common', 'scripts'))
from lakebase import connect, run_sql

SCHEMA = os.environ.get('LB_SCHEMA', 'factory')
print('connected; schema:', SCHEMA)

## Step 1 — Inspect the operation table

See the structure of operations (work routing on machines). The SQL:

```sql
SELECT op_id, machine_id, starts_at, ends_at FROM factory.operation LIMIT 3
```

In [ ]:
sql = "SELECT op_id, machine_id, starts_at, ends_at FROM factory.operation LIMIT 3"
rows = run_sql(sql)
print("Sample operations:")
for row in rows:
    print(f"  {row}")

## Step 2 — Create GiST exclusion constraint

Enable the btree_gist extension, then add an exclusion constraint that prevents overlapping time ranges for the same machine. The SQL:

```sql
CREATE EXTENSION IF NOT EXISTS btree_gist;
ALTER TABLE factory.operation
  ADD CONSTRAINT IF NOT EXISTS no_machine_overlap
  EXCLUDE USING gist (machine_id WITH =,
                      tstzrange(starts_at, ends_at) WITH &&)
```

In [ ]:
with connect() as conn:
    with conn.cursor() as cur:
        try:
            cur.execute("CREATE EXTENSION IF NOT EXISTS btree_gist")
            print("btree_gist extension created")
        except:
            pass

        sql = """
        ALTER TABLE factory.operation
          ADD CONSTRAINT IF NOT EXISTS no_machine_overlap
          EXCLUDE USING gist (machine_id WITH =,
                              tstzrange(starts_at, ends_at) WITH &&)
        """
        try:
            cur.execute(sql)
            print("Exclusion constraint created: no_machine_overlap")
        except Exception as e:
            print(f"Constraint may already exist: {e}")

## Step 3 — Attempt an overlapping insert

Try to insert an operation that overlaps with an existing one on the same machine. The constraint should reject it.

In [ ]:
import datetime

# Get an existing operation to create an overlap
ops = run_sql("""
SELECT op_id, machine_id, starts_at, ends_at
  FROM factory.operation
 WHERE machine_id IS NOT NULL AND starts_at IS NOT NULL AND ends_at IS NOT NULL
 LIMIT 1
""")

if not ops:
    print("No existing operations found")
else:
    op_id, machine_id, starts_at, ends_at = ops[0]

    # Try to insert an overlapping operation
    with connect() as conn:
        with conn.cursor() as cur:
            try:
                # Create an operation that overlaps
                overlap_start = starts_at
                overlap_end = ends_at + datetime.timedelta(hours=1)

                cur.execute("""
                INSERT INTO factory.operation
                  (op_id, wo_id, machine_id, seq, starts_at, ends_at, status)
                VALUES
                  (%s, %s, %s, %s, %s, %s, %s)
                """, ('OP-OVERLAP', 'WO-TEST', machine_id, 1, overlap_start, overlap_end, 'pending'))
                print("ERROR: Overlap should have been rejected!")
            except Exception as e:
                error_msg = str(e)
                if "no_machine_overlap" in error_msg:
                    print("Good: Overlap rejected by constraint")
                    print(f"  {error_msg[:100]}...")
                else:
                    print(f"Error: {error_msg[:100]}...")

## Step 4 — EXPLAIN shows constraint check

Run EXPLAIN to see the constraint check in the query plan.

In [ ]:
sql = "EXPLAIN INSERT INTO factory.operation VALUES ('test', 'wo', 'M-204', 1, now(), now() + interval '1 hour', 'pending')"

with connect() as conn:
    with conn.cursor() as cur:
        cur.execute(sql)
        rows = cur.fetchall()
        print("Query plan (should mention constraint):")
        for row in rows:
            print(f"  {row[0]}")

## What you learned

You created a GiST exclusion constraint that prevents overlapping time ranges for the same machine. This constraint is checked at the database level, not the application level, so no double-booking is possible.

Next: **Watch the planner** uses EXPLAIN ANALYZE to understand when indexes matter.